In [1]:
import csv
import os
from datetime import datetime
import pandas as pd


def detect_column_type(values):

    number_count = 0
    date_count = 0
    text_count = 0
    alphanumeric_count = 0

    for value in values:

        value = value.strip()

        if value == "":
            continue

        try:
            float(value)
            number_count += 1
            continue
        except:
            pass

        try:
            datetime.strptime(value, "%Y-%m-%d")
            date_count += 1
            continue
        except:
            pass

        if value.isalpha():
            text_count += 1

        elif value.isalnum():
            alphanumeric_count += 1

        else:
            text_count += 1

    counts = {
        "NUMBER": number_count,
        "DATE": date_count,
        "TEXT": text_count,
        "ALPHANUMERIC": alphanumeric_count
    }

    return max(counts, key=counts.get)


def is_valid(value, data_type):

    value = value.strip()

    if value == "":
        return False

    if data_type == "NUMBER":

        try:
            return float(value) > 0
        except:
            return False

    if data_type == "DATE":

        try:
            datetime.strptime(value, "%Y-%m-%d")
            return True
        except:
            return False

    if data_type == "TEXT":

        return value.replace(" ", "").isalpha()

    if data_type == "ALPHANUMERIC":

        return value.isalnum()

    return False


def clean_file(file_path):

    with open(file_path, "r", newline="", encoding="utf-8") as file:

        data = csv.DictReader(file)

        rows = list(data)

    if not rows:
        return [], []

    column_types = {}

    for column in rows[0]:

        values = []

        for row in rows:
            values.append(row[column])

        column_types[column] = detect_column_type(values)

    clean_data = []
    bad_data = []

    seen_rows = set()

    for row in rows:

        row_tuple = tuple(row.values())

        # Duplicate check
        if row_tuple in seen_rows:
            bad_data.append(row)
            continue

        seen_rows.add(row_tuple)

        valid = True

        # Validation
        for column in row:

            value = row[column]

            data_type = column_types[column]

            if not is_valid(value, data_type):

                valid = False
                break

        if valid:
            clean_data.append(row)
        else:
            bad_data.append(row)

    return clean_data, bad_data


# -----------------------------------------
# FIND CSV AUTOMATICALLY
# -----------------------------------------

raw_folder = "../data/raw"

csv_files = []

for file in os.listdir(raw_folder):

    if file.endswith(".csv"):
        csv_files.append(file)


# -----------------------------------------
# PROCESS CSV FILES
# -----------------------------------------

for file_name in csv_files:

    file_path = os.path.join(raw_folder, file_name)

    print("\nProcessing:", file_name)

    clean_data, bad_data = clean_file(file_path)

    clean_df = pd.DataFrame(clean_data)
    bad_df = pd.DataFrame(bad_data)

    print("Clean rows:", len(clean_df))
    print("Bad rows:", len(bad_df))

    # Create folders
    os.makedirs("../data/processed", exist_ok=True)
    os.makedirs("../data/quarantine", exist_ok=True)

    # Get original filename without .csv
    name = os.path.splitext(file_name)[0]

    # Save clean data
    clean_df.to_csv(
        "../data/processed/" + name + "_clean.csv",
        index=False
    )

    # Save bad data
    bad_df.to_csv(
        "../data/quarantine/" + name + "_bad.csv",
        index=False
    )

    print("Clean data saved to:", "../data/processed/" + name + "_clean.csv")
    print("Bad data saved to:", "../data/quarantine/" + name + "_bad.csv")


Processing: employee.csv
Clean rows: 3
Bad rows: 8
Clean data saved to: ../data/processed/employee_clean.csv
Bad data saved to: ../data/quarantine/employee_bad.csv

Processing: inventory.csv
Clean rows: 5
Bad rows: 8
Clean data saved to: ../data/processed/inventory_clean.csv
Bad data saved to: ../data/quarantine/inventory_bad.csv

Processing: students.csv
Clean rows: 4
Bad rows: 9
Clean data saved to: ../data/processed/students_clean.csv
Bad data saved to: ../data/quarantine/students_bad.csv

Processing: transactions.csv
Clean rows: 3
Bad rows: 8
Clean data saved to: ../data/processed/transactions_clean.csv
Bad data saved to: ../data/quarantine/transactions_bad.csv
